# Urban Occlusion-Aware Segmentation - Exploration

This notebook provides exploratory analysis and visualization of the occlusion-aware segmentation model.

In [ ]:
import sys
from pathlib import Path

# Add project to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from urban_occlusion_aware_segmentation.data.loader import get_data_loaders
from urban_occlusion_aware_segmentation.models.model import build_model
from urban_occlusion_aware_segmentation.utils.config import load_config, get_device, set_seed

%matplotlib inline

## 1. Load Configuration and Setup

In [ ]:
# Load config
config = load_config("../configs/default.yaml")

# Set seed
set_seed(42)

# Get device
device = get_device("auto")
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
# Create data loaders
train_loader, val_loader = get_data_loaders(config)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Get a sample batch
sample_images, sample_masks = next(iter(train_loader))
print(f"Image shape: {sample_images.shape}")
print(f"Mask shape: {sample_masks.shape}")

## 3. Visualize Data Samples

In [ ]:
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Denormalize image for visualization."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

# Visualize samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(4, sample_images.shape[0])):
    # Image
    img = denormalize(sample_images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Image {i+1}")
    axes[0, i].axis("off")
    
    # Mask
    mask = sample_masks[i].numpy()
    axes[1, i].imshow(mask, cmap="tab20")
    axes[1, i].set_title(f"Segmentation {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 4. Build and Inspect Model

In [ ]:
# Build model
model = build_model(config, device)

# Model summary
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {num_trainable:,}")
print(f"Model type: {config['model']['type']}")

## 5. Test Forward Pass

In [ ]:
model.eval()

with torch.no_grad():
    sample_images_device = sample_images.to(device)
    
    if config['model']['type'] == 'ensemble':
        outputs, uncertainty = model(sample_images_device, return_uncertainty=True)
        print(f"Output shape: {outputs.shape}")
        print(f"Uncertainty shape: {uncertainty.shape}")
    else:
        outputs = model(sample_images_device)
        print(f"Output shape: {outputs.shape}")
        uncertainty = None

# Get predictions
predictions = outputs.argmax(dim=1)
print(f"Predictions shape: {predictions.shape}")

## 6. Visualize Predictions with Uncertainty

In [ ]:
if uncertainty is not None:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    
    for i in range(min(4, sample_images.shape[0])):
        # Image
        img = denormalize(sample_images[i]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Image {i+1}")
        axes[0, i].axis("off")
        
        # Prediction
        pred = predictions[i].cpu().numpy()
        axes[1, i].imshow(pred, cmap="tab20")
        axes[1, i].set_title(f"Prediction {i+1}")
        axes[1, i].axis("off")
        
        # Uncertainty
        unc = uncertainty[i].cpu().numpy()
        im = axes[2, i].imshow(unc, cmap="hot")
        axes[2, i].set_title(f"Uncertainty {i+1}")
        axes[2, i].axis("off")
        plt.colorbar(im, ax=axes[2, i], fraction=0.046)
    
    plt.tight_layout()
    plt.show()
else:
    print("Uncertainty not available for this model type.")

## 7. Analyze Boundary Regions

In [ ]:
from urban_occlusion_aware_segmentation.data.preprocessing import compute_boundary_mask

# Compute boundaries for ground truth
idx = 0
mask_np = sample_masks[idx].numpy()
boundary = compute_boundary_mask(mask_np, width=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original mask
axes[0].imshow(mask_np, cmap="tab20")
axes[0].set_title("Ground Truth Segmentation")
axes[0].axis("off")

# Boundary
axes[1].imshow(boundary, cmap="gray")
axes[1].set_title("Extracted Boundaries")
axes[1].axis("off")

# Overlay
img = denormalize(sample_images[idx]).permute(1, 2, 0).numpy()
img = np.clip(img, 0, 1)
axes[2].imshow(img)
axes[2].imshow(boundary, cmap="Reds", alpha=0.5)
axes[2].set_title("Boundaries Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 8. Class Distribution Analysis

In [ ]:
# Analyze class distribution
all_masks = []
for images, masks in train_loader:
    all_masks.append(masks.numpy().flatten())
    if len(all_masks) >= 10:  # Sample first 10 batches
        break

all_masks = np.concatenate(all_masks)
unique, counts = np.unique(all_masks, return_counts=True)

plt.figure(figsize=(12, 6))
plt.bar(unique, counts)
plt.xlabel("Class ID")
plt.ylabel("Pixel Count")
plt.title("Class Distribution in Training Data")
plt.yscale("log")
plt.grid(axis="y", alpha=0.3)
plt.show()

print("Class distribution:")
for cls_id, count in zip(unique, counts):
    print(f"Class {cls_id}: {count:,} pixels ({100*count/counts.sum():.2f}%)")

## 9. Conclusion

This notebook demonstrated:
- Data loading and visualization
- Model architecture inspection
- Forward pass with uncertainty quantification
- Boundary detection for occlusion-aware segmentation
- Class distribution analysis

Next steps:
1. Run full training with `python scripts/train.py`
2. Evaluate on test data with `python scripts/evaluate.py --checkpoint checkpoints/best_model.pth`
3. Analyze results and tune hyperparameters